In [ ]:
# Google Colab : télécharger les données de cette séance / download this session's data
import os, urllib.request
BASE = "https://raw.githubusercontent.com/Marzoug-Nabil/Nabil/main/academic/teaching/fsci6222/materials/s05/"
for f in ['temps_attente.csv']:
    os.makedirs('data', exist_ok=True)
    for dst in (f, os.path.join('data', f)):
        if not os.path.exists(dst):
            urllib.request.urlretrieve(BASE + f, dst)


# FSCI 6222 — Analyse de données
## Séance 5 : Probabilités et distributions

**Enseignant :** Nabil Marzoug  
**Date :** 26 mai 2026  
**Université de Moncton — Faculté des sciences**

---

### Objectifs de la séance
1. Comprendre les concepts fondamentaux de probabilité
2. Distinguer les principales distributions de probabilité
3. Utiliser scipy.stats pour travailler avec les distributions
4. Appliquer la loi normale et le théorème central limite
5. Calculer des probabilités et des intervalles de confiance

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuration des graphiques
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Librairies chargées avec succès!")

---
# Partie 1 — Concepts fondamentaux de probabilité

## 1.1 Qu'est-ce qu'une probabilité?

La **probabilité** d'un événement est un nombre entre 0 et 1 qui mesure la chance que cet événement se produise.

- $P(A) = 0$ → l'événement A est **impossible**
- $P(A) = 1$ → l'événement A est **certain**
- $P(A) = 0.5$ → l'événement A a une chance sur deux de se produire

**Formule classique :**

$$P(A) = \frac{\text{nombre de cas favorables}}{\text{nombre de cas possibles}}$$

In [ ]:
# Exemple : lancer un dé
# Probabilité d'obtenir un 6
p_six = 1 / 6
print(f"P(obtenir un 6) = {p_six:.4f} = {p_six*100:.2f}%")

# Probabilité d'obtenir un nombre pair (2, 4, 6)
p_pair = 3 / 6
print(f"P(nombre pair) = {p_pair:.4f} = {p_pair*100:.2f}%")

# Probabilité d'obtenir un nombre > 4 (5, 6)
p_sup4 = 2 / 6
print(f"P(nombre > 4) = {p_sup4:.4f} = {p_sup4*100:.2f}%")

## 1.2 Simulation et loi des grands nombres

La **loi des grands nombres** stipule que plus on répète une expérience, plus la fréquence observée se rapproche de la probabilité théorique.

In [ ]:
# Simuler des lancers de dé
np.random.seed(42)

# Lancer un dé N fois
n_lancers = [10, 100, 1000, 10000, 100000]

print("Fréquence d'obtenir un 6 :")
print(f"{'N lancers':>10} | {'Fréquence':>10} | {'P théorique':>12}")
print("-" * 40)

for n in n_lancers:
    lancers = np.random.randint(1, 7, n)
    freq_six = np.sum(lancers == 6) / n
    print(f"{n:>10} | {freq_six:>10.4f} | {1/6:>12.4f}")

In [ ]:
# Visualiser la convergence
np.random.seed(42)
lancers = np.random.randint(1, 7, 10000)

# Fréquence cumulée de 6
est_six = (lancers == 6).astype(float)
freq_cumul = np.cumsum(est_six) / np.arange(1, 10001)

plt.figure(figsize=(10, 5))
plt.plot(freq_cumul, color='steelblue', linewidth=0.8)
plt.axhline(y=1/6, color='red', linestyle='--', label=f'P théorique = {1/6:.4f}')
plt.xlabel('Nombre de lancers')
plt.ylabel('Fréquence de 6')
plt.title('Loi des grands nombres — Convergence vers P(6) = 1/6')
plt.legend()
plt.tight_layout()
plt.show()

## 1.3 Règles de probabilité

In [ ]:
# Complément : P(A') = 1 - P(A)
p_six = 1/6
p_pas_six = 1 - p_six
print(f"P(pas un 6) = 1 - {p_six:.4f} = {p_pas_six:.4f}")

# Union (OU) pour événements mutuellement exclusifs : P(A ou B) = P(A) + P(B)
p_1ou6 = 1/6 + 1/6
print(f"P(1 ou 6) = {p_1ou6:.4f}")

# Intersection (ET) pour événements indépendants : P(A et B) = P(A) × P(B)
# Probabilité de 2 six consécutifs
p_deux_six = (1/6) * (1/6)
print(f"P(deux 6 consécutifs) = {p_deux_six:.4f}")

# Union pour événements NON mutuellement exclusifs : P(A ou B) = P(A) + P(B) - P(A et B)
# Tirer une carte : P(rouge ou roi)
p_rouge = 26/52
p_roi = 4/52
p_roi_rouge = 2/52
p_rouge_ou_roi = p_rouge + p_roi - p_roi_rouge
print(f"P(rouge ou roi) = {p_rouge_ou_roi:.4f}")

### Exercice 1 — Probabilités de base

Un sac contient 5 billes rouges, 3 billes bleues et 2 billes vertes.

1. Calculer la probabilité de tirer une bille rouge
2. Calculer la probabilité de tirer une bille qui n'est PAS verte
3. On tire 2 billes avec remise. Calculer la probabilité d'obtenir 2 billes rouges
4. Simuler 10 000 tirages et vérifier que la fréquence converge vers P(rouge)
5. Visualiser la convergence sur un graphique

In [ ]:
# 1. P(rouge)
p_rouge = 5 / 10
print(f"P(rouge) = {p_rouge}")

# 2. P(pas verte) = 1 - P(verte)
p_pas_verte = 1 - 2/10
print(f"P(pas verte) = {p_pas_verte}")

# 3. P(2 rouges avec remise) = P(rouge) × P(rouge)
p_2rouges = p_rouge ** 2
print(f"P(2 rouges) = {p_2rouges}")

# 4. Simulation
np.random.seed(42)
billes = ['R']*5 + ['B']*3 + ['V']*2
tirages = np.random.choice(billes, 10000)
freq_rouge = np.sum(tirages == 'R') / 10000
print(f"Fréquence simulée : {freq_rouge:.4f}, Théorique : {p_rouge}")

# 5. Graphique
est_rouge = (tirages == 'R').astype(float)
freq_cumul = np.cumsum(est_rouge) / np.arange(1, 10001)

plt.figure(figsize=(10, 5))
plt.plot(freq_cumul, color='red', linewidth=0.8)
plt.axhline(y=0.5, color='black', linestyle='--', label='P théorique = 0.5')
plt.xlabel('Nombre de tirages')
plt.ylabel('Fréquence de rouge')
plt.title('Convergence vers P(rouge) = 0.5')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
np.cumsum((tirages == 'R').astype('int'))/np.arange(1, 10001)

<details>
<summary> Cliquer pour voir la solution</summary>

```python
# 1. P(rouge)
p_rouge = 5 / 10
print(f"P(rouge) = {p_rouge}")

# 2. P(pas verte) = 1 - P(verte)
p_pas_verte = 1 - 2/10
print(f"P(pas verte) = {p_pas_verte}")

# 3. P(2 rouges avec remise) = P(rouge) × P(rouge)
p_2rouges = p_rouge ** 2
print(f"P(2 rouges) = {p_2rouges}")

# 4. Simulation
np.random.seed(42)
billes = ['R']*5 + ['B']*3 + ['V']*2
tirages = np.random.choice(billes, 10000)
freq_rouge = np.sum(tirages == 'R') / 10000
print(f"Fréquence simulée : {freq_rouge:.4f}, Théorique : {p_rouge}")

# 5. Graphique
est_rouge = (tirages == 'R').astype(float)
freq_cumul = np.cumsum(est_rouge) / np.arange(1, 10001)

plt.figure(figsize=(10, 5))
plt.plot(freq_cumul, color='red', linewidth=0.8)
plt.axhline(y=0.5, color='black', linestyle='--', label='P théorique = 0.5')
plt.xlabel('Nombre de tirages')
plt.ylabel('Fréquence de rouge')
plt.title('Convergence vers P(rouge) = 0.5')
plt.legend()
plt.tight_layout()
plt.show()
```

</details>

---
# Partie 2 — Variables aléatoires et distributions

Une **variable aléatoire** est une variable dont la valeur dépend du hasard. Elle peut être :
- **Discrète** : prend des valeurs isolées (entiers) — ex: nombre de clients, résultat d'un dé
- **Continue** : prend n'importe quelle valeur dans un intervalle — ex: taille, température, temps d'attente

Une **distribution de probabilité** décrit comment les probabilités sont réparties sur les valeurs possibles.

## 2.1 scipy.stats — La boîte à outils

Le module `scipy.stats` fournit des outils pour toutes les distributions. Chaque distribution offre les mêmes méthodes :

| Méthode | Description |
|---|---|
| `rvs(size=n)` | Générer n valeurs aléatoires |
| `pmf(x)` / `pdf(x)` | Probabilité en un point (discrète / continue) |
| `cdf(x)` | Probabilité cumulée P(X ≤ x) |
| `ppf(q)` | Quantile (inverse de cdf) |
| `mean()` | Espérance (moyenne théorique) |
| `std()` | Écart-type théorique |

---
# Partie 3 — Distributions discrètes

## 3.1 Distribution de Bernoulli

Modélise une expérience avec **deux résultats possibles** : succès (1) ou échec (0).

- Paramètre : $p$ = probabilité de succès
- Exemples : lancer une pièce, réussir un examen, un client achète ou non

In [ ]:
from scipy.stats import bernoulli

# Paramètre : p = probabilité de succès
p = 0.7  # 70% de chance de réussite

# Probabilités
print(f"P(X = 0) = P(échec) = {bernoulli.pmf(0, p):.2f}")
print(f"P(X = 1) = P(succès) = {bernoulli.pmf(1, p):.2f}")

# Moyenne et écart-type
print(f"Moyenne : {bernoulli.mean(p):.2f}")
print(f"Écart-type : {bernoulli.std(p):.4f}")

# Simulation : 20 étudiants passent un examen avec 70% de réussite
np.random.seed(42)
resultats = bernoulli.rvs(p, size=20)
print(f"\nRésultats : {resultats}")
print(f"Réussites : {np.sum(resultats)}/20 = {np.mean(resultats)*100:.0f}%")

## 3.2 Distribution binomiale

Modélise le **nombre de succès** sur $n$ essais indépendants de Bernoulli.

- Paramètres : $n$ = nombre d'essais, $p$ = probabilité de succès
- $X \sim B(n, p)$
- Moyenne : $\mu = n \times p$
- Écart-type : $\sigma = \sqrt{n \times p \times (1-p)}$

In [ ]:
from scipy.stats import binom

# Exemple : 20 questions à un examen, P(bonne réponse) = 0.6
n = 20
p = 0.6

# Probabilités spécifiques
print(f"P(exactement 12 bonnes réponses) = {binom.pmf(12, n, p):.4f}")
print(f"P(au plus 10 bonnes réponses) = {binom.cdf(10, n, p):.4f}")
print(f"P(au moins 15 bonnes réponses) = {1 - binom.cdf(14, n, p):.4f}")

# Moyenne et écart-type
print(f"\nMoyenne : {binom.mean(n, p):.1f}")
print(f"Écart-type : {binom.std(n, p):.2f}")

In [ ]:
# Visualiser la distribution binomiale
x = np.arange(0, n+1)
probabilites = binom.pmf(x, n, p)

plt.figure(figsize=(10, 5))
plt.bar(x, probabilites, color='steelblue', edgecolor='white')
plt.axvline(x=binom.mean(n, p), color='red', linestyle='--', 
            label=f'Moyenne = {binom.mean(n, p):.1f}')
plt.xlabel('Nombre de bonnes réponses')
plt.ylabel('Probabilité')
plt.title(f'Distribution Binomiale B(n={n}, p={p})')
plt.legend()
plt.tight_layout()
plt.show()

## 3.3 Distribution de Poisson

Modélise le **nombre d'événements** survenant dans un intervalle de temps ou d'espace fixé.

- Paramètre : $\lambda$ = nombre moyen d'événements par intervalle
- $X \sim Poisson(\lambda)$
- Moyenne = Variance = $\lambda$
- Exemples : nombre d'appels par heure, nombre d'erreurs par page, nombre de clients par jour

In [ ]:
from scipy.stats import poisson

# Exemple : un service client reçoit en moyenne 5 appels par heure
lam = 5

print(f"P(exactement 3 appels) = {poisson.pmf(3, lam):.4f}")
print(f"P(au plus 3 appels) = {poisson.cdf(3, lam):.4f}")
print(f"P(plus de 7 appels) = {1 - poisson.cdf(7, lam):.4f}")
print(f"P(aucun appel) = {poisson.pmf(0, lam):.4f}")

print(f"\nMoyenne : {poisson.mean(lam):.1f}")
print(f"Écart-type : {poisson.std(lam):.2f}")

In [ ]:
# Comparer plusieurs distributions de Poisson
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, lam in zip(axes, [2, 5, 10]):
    x = np.arange(0, lam*4)
    ax.bar(x, poisson.pmf(x, lam), color='steelblue', edgecolor='white')
    ax.set_title(f'Poisson(λ={lam})')
    ax.set_xlabel('k')
    ax.set_ylabel('P(X=k)')

plt.suptitle('Distributions de Poisson pour différentes valeurs de λ', fontsize=14)
plt.tight_layout()
plt.show()

### Exercice 2 — Distributions discrètes

**Partie A — Binomiale :**
Un examen contient 30 questions à choix multiples (4 choix par question). Un étudiant répond au hasard.
1. Quelle est la probabilité qu'il obtienne exactement 10 bonnes réponses?
2. Quelle est la probabilité qu'il réussisse (≥ 18 bonnes réponses)?
3. Combien de bonnes réponses peut-il espérer en moyenne?
4. Visualiser la distribution

**Partie B — Poisson :**
Un site web reçoit en moyenne 8 visiteurs par minute.
1. Quelle est la probabilité d'avoir exactement 5 visiteurs en une minute?
2. Quelle est la probabilité d'avoir plus de 12 visiteurs?
3. Simuler le nombre de visiteurs par minute pendant 1 heure (60 min) et tracer un histogramme

In [ ]:
# Partie A — Binomiale
# Votre code ici


In [ ]:
# Partie B — Poisson


<details>
<summary> Cliquer pour voir la solution</summary>

```python
# Partie A
n, p = 30, 1/4

# 1.
print(f"P(X = 10) = {binom.pmf(10, n, p):.4f}")

# 2.
print(f"P(X >= 18) = {1 - binom.cdf(17, n, p):.6f}")

# 3.
print(f"Espérance = {n * p:.1f} bonnes réponses")

# 4.
x = np.arange(0, n+1)
plt.bar(x, binom.pmf(x, n, p), color='steelblue', edgecolor='white')
plt.axvline(x=n*p, color='red', linestyle='--', label=f'Moyenne = {n*p}')
plt.xlabel('Nombre de bonnes réponses')
plt.ylabel('Probabilité')
plt.title(f'B(n={n}, p={p:.2f}) — Réponse au hasard')
plt.legend()
plt.show()

# Partie B
lam = 8

# 1.
print(f"P(X = 5) = {poisson.pmf(5, lam):.4f}")

# 2.
print(f"P(X > 12) = {1 - poisson.cdf(12, lam):.4f}")

# 3.
simulation = poisson.rvs(lam, size=60)
plt.hist(simulation, bins=range(0, 20), color='steelblue', edgecolor='white')
plt.axvline(x=np.mean(simulation), color='red', linestyle='--', label=f'Moyenne = {np.mean(simulation):.1f}')
plt.xlabel('Visiteurs par minute')
plt.ylabel('Fréquence')
plt.title('Simulation Poisson(λ=8) — 60 minutes')
plt.legend()
plt.show()
```

</details>

---
# Partie 4 — Distributions continues

## 4.1 Distribution uniforme

Tous les intervalles de même longueur ont la même probabilité. C'est la distribution du "hasard pur".

- Paramètres : $a$ (min), $b$ (max)
- Moyenne : $(a + b) / 2$
- Exemples : générateur de nombres aléatoires, angle d'une roulette

In [ ]:
from scipy.stats import uniform

# Uniforme entre 0 et 10
a, b = 0, 10

# P(3 < X < 7)
p = uniform.cdf(7, loc=a, scale=b-a) - uniform.cdf(3, loc=a, scale=b-a)
print(f"P(3 < X < 7) = {p:.2f}")

# Simulation et visualisation
np.random.seed(42)
echantillon = uniform.rvs(loc=a, scale=b-a, size=10000)

plt.figure(figsize=(10, 5))
plt.hist(echantillon, bins=50, density=True, color='steelblue', edgecolor='white', alpha=0.7)
x = np.linspace(a-1, b+1, 100)
plt.plot(x, uniform.pdf(x, loc=a, scale=b-a), 'r-', linewidth=2, label='PDF théorique')
plt.xlabel('Valeur')
plt.ylabel('Densité')
plt.title(f'Distribution Uniforme U({a}, {b})')
plt.legend()
plt.tight_layout()
plt.show()

## 4.2 Distribution normale (gaussienne)

La distribution la plus importante en statistiques. Sa forme est une **courbe en cloche** symétrique.

- Paramètres : $\mu$ (moyenne), $\sigma$ (écart-type)
- $X \sim N(\mu, \sigma)$
- **Règle empirique (68-95-99.7)** :
  - 68% des valeurs dans $[\mu - \sigma, \mu + \sigma]$
  - 95% des valeurs dans $[\mu - 2\sigma, \mu + 2\sigma]$
  - 99.7% des valeurs dans $[\mu - 3\sigma, \mu + 3\sigma]$

In [ ]:
from scipy.stats import norm

# Distribution normale : notes d'examen
mu = 75     # moyenne
sigma = 10  # écart-type

# Probabilités
print(f"P(X < 60) = {norm.cdf(60, mu, sigma):.4f}")
print(f"P(X > 90) = {1 - norm.cdf(90, mu, sigma):.4f}")
print(f"P(65 < X < 85) = {norm.cdf(85, mu, sigma) - norm.cdf(65, mu, sigma):.4f}")

In [ ]:
# Visualiser la distribution normale avec la règle 68-95-99.7
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
y = norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x, y, 'k-', linewidth=2)

# 99.7%
x_fill = np.linspace(mu - 3*sigma, mu + 3*sigma, 1000)
ax.fill_between(x_fill, norm.pdf(x_fill, mu, sigma), alpha=0.15, color='blue', label='99.7% (±3σ)')

# 95%
x_fill = np.linspace(mu - 2*sigma, mu + 2*sigma, 1000)
ax.fill_between(x_fill, norm.pdf(x_fill, mu, sigma), alpha=0.25, color='blue', label='95% (±2σ)')

# 68%
x_fill = np.linspace(mu - sigma, mu + sigma, 1000)
ax.fill_between(x_fill, norm.pdf(x_fill, mu, sigma), alpha=0.4, color='blue', label='68% (±1σ)')

ax.axvline(x=mu, color='red', linestyle='--', label=f'μ = {mu}')
ax.set_xlabel('Notes')
ax.set_ylabel('Densité')
ax.set_title(f'Distribution Normale N(μ={mu}, σ={sigma}) — Règle 68-95-99.7')
ax.legend()
plt.tight_layout()
plt.show()

## 4.3 Z-score (score standard)

Le **z-score** mesure à combien d'écarts-types une valeur se trouve de la moyenne :

$$z = \frac{x - \mu}{\sigma}$$

- $z = 0$ → la valeur est exactement à la moyenne
- $z = 1$ → la valeur est à 1 écart-type au-dessus de la moyenne
- $z = -2$ → la valeur est à 2 écarts-types en dessous

In [ ]:
mu, sigma = 75, 10

# Z-score pour différentes notes
notes_test = [60, 75, 85, 95]

print(f"{'Note':>6} | {'Z-score':>8} | {'Percentile':>12} | {'Interprétation'}")
print("-" * 60)

for note in notes_test:
    z = (note - mu) / sigma
    percentile = norm.cdf(note, mu, sigma) * 100
    print(f"{note:>6} | {z:>+8.2f} | {percentile:>11.1f}% | ", end="")
    
    if abs(z) < 1:
        print("Dans la norme")
    elif abs(z) < 2:
        print("Modérément extrême")
    else:
        print("Très extrême")

In [ ]:
# Application : standardiser un dataset
np.random.seed(42)
notes = np.random.normal(mu, sigma, 100)

# Calculer les z-scores
z_scores = (notes - np.mean(notes)) / np.std(notes)

# Ou avec scipy
z_scores_scipy = stats.zscore(notes)

print(f"Moyenne des z-scores : {np.mean(z_scores):.6f} (≈ 0)")
print(f"Écart-type des z-scores : {np.std(z_scores):.6f} (≈ 1)")

# Identifier les valeurs aberrantes (|z| > 2)
aberrantes = notes[np.abs(z_scores) > 2]
print(f"\nValeurs aberrantes (|z| > 2) : {len(aberrantes)}")
print(aberrantes.round(1))

## 4.4 Distribution exponentielle

Modélise le **temps d'attente** entre deux événements successifs d'un processus de Poisson.

- Paramètre : $\lambda$ = taux moyen d'événements
- Moyenne : $1/\lambda$
- Exemples : temps entre deux appels, durée de vie d'un composant, temps d'attente au guichet

In [ ]:
from scipy.stats import expon

# Temps d'attente moyen au guichet = 5 minutes → λ = 1/5
lam = 1/5
scale = 1/lam   # scipy utilise scale = 1/λ

print(f"P(attente < 3 min) = {expon.cdf(3, scale=scale):.4f}")
print(f"P(attente > 10 min) = {1 - expon.cdf(10, scale=scale):.4f}")
print(f"P(3 < attente < 8 min) = {expon.cdf(8, scale=scale) - expon.cdf(3, scale=scale):.4f}")

# Visualiser
x = np.linspace(0, 25, 1000)

plt.figure(figsize=(10, 5))
plt.plot(x, expon.pdf(x, scale=scale), 'b-', linewidth=2, label='PDF')
plt.fill_between(x[x <= 10], expon.pdf(x[x <= 10], scale=scale), alpha=0.3, color='steelblue')
plt.axvline(x=scale, color='red', linestyle='--', label=f'Moyenne = {scale} min')
plt.xlabel('Temps d\'attente (minutes)')
plt.ylabel('Densité')
plt.title(f'Distribution Exponentielle (moyenne = {scale} min)')
plt.legend()
plt.tight_layout()
plt.show()

### Exercice 3 — Distributions continues

**Partie A — Normale :**
Les tailles des étudiants suivent une loi normale $N(170, 8)$ cm.
1. Quelle est la probabilité qu'un étudiant mesure plus de 185 cm?
2. Quelle est la probabilité qu'un étudiant mesure entre 160 et 180 cm?
3. Quelle taille correspond au 90e percentile?
4. Simuler 1000 tailles et tracer l'histogramme avec la courbe théorique

**Partie B — Application sur un dataset réel :**
Charger `data/temps_attente.csv` et :
1. Calculer la moyenne et l'écart-type des temps d'attente
2. Tracer l'histogramme des temps d'attente
3. Tester si les données suivent une distribution normale (qq-plot)
4. Tester si les données suivent une distribution exponentielle

In [ ]:
# Partie A — Normale

# Votre code ici


In [ ]:
# Partie B — Dataset réel

# Votre code ici


<details>
<summary> Cliquer pour voir la solution</summary>

```python
# Partie A
mu, sigma = 170, 8

# 1.
print(f"P(X > 185) = {1 - norm.cdf(185, mu, sigma):.4f}")

# 2.
print(f"P(160 < X < 180) = {norm.cdf(180, mu, sigma) - norm.cdf(160, mu, sigma):.4f}")

# 3.
p90 = norm.ppf(0.90, mu, sigma)
print(f"90e percentile = {p90:.1f} cm")

# 4.
np.random.seed(42)
tailles = norm.rvs(mu, sigma, size=1000)
x = np.linspace(140, 200, 1000)

plt.figure(figsize=(10, 5))
plt.hist(tailles, bins=30, density=True, color='steelblue', edgecolor='white', alpha=0.7)
plt.plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Courbe théorique')
plt.xlabel('Taille (cm)')
plt.ylabel('Densité')
plt.title(f'Simulation N({mu}, {sigma}) — 1000 étudiants')
plt.legend()
plt.show()

# Partie B
df = pd.read_csv('data/temps_attente.csv')

# 1.
moy = df['temps_attente_min'].mean()
std = df['temps_attente_min'].std()
print(f"Moyenne : {moy:.2f} min")
print(f"Écart-type : {std:.2f} min")

# 2.
plt.figure(figsize=(10, 5))
plt.hist(df['temps_attente_min'], bins=15, density=True, color='steelblue', edgecolor='white')
plt.xlabel('Temps d\'attente (min)')
plt.ylabel('Densité')
plt.title('Distribution des temps d\'attente')
plt.show()

# 3. QQ-plot pour normalité
fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df['temps_attente_min'], dist='norm', plot=ax)
ax.set_title('QQ-Plot (test de normalité)')
plt.show()

# 4. QQ-plot pour exponentielle
fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df['temps_attente_min'], dist='expon', plot=ax)
ax.set_title('QQ-Plot (test exponentielle)')
plt.show()
```

</details>

---
# Partie 5 — Théorème central limite (TCL)

## 5.1 Énoncé du théorème

Le **théorème central limite** stipule que la distribution des **moyennes d'échantillons** tend vers une distribution normale quand la taille de l'échantillon augmente, **quelle que soit** la distribution d'origine.

Si $X_1, X_2, ..., X_n$ sont des variables aléatoires i.i.d. avec moyenne $\mu$ et écart-type $\sigma$, alors :

$$\bar{X}_n \sim N\left(\mu, \frac{\sigma}{\sqrt{n}}\right) \text{ quand } n \to \infty$$

En pratique, $n \geq 30$ suffit généralement.

## Démonstration visuelle du Théorème Central Limite

Cette démonstration illustre concrètement le TCL : nous allons partir d'une distribution **très asymétrique** (exponentielle) et observer comment la distribution **des moyennes d'échantillons** se transforme progressivement en courbe normale.

### Le principe de l'expérience

Le code reproduit numériquement la définition même du TCL, en quatre étapes :

1. **On choisit une population d'origine** : la loi **exponentielle** de moyenne 5 (`np.random.exponential(5, n)`). C'est une distribution fortement asymétrique, avec beaucoup de petites valeurs et une longue queue à droite — l'opposé d'une courbe normale.

2. **On prélève un échantillon de taille `n`** et on calcule sa **moyenne**. Cela produit **un seul** nombre.

3. **On répète l'opération 10 000 fois** (`n_simulations`). On obtient ainsi 10 000 moyennes d'échantillons :
`moyennes = [np.mean(np.random.exponential(5, n)) for _ in range(n_simulations)]`

4. **On trace l'histogramme de ces 10 000 moyennes**, et on recommence pour quatre tailles d'échantillon différentes : `n = 1, 5, 30, 100`.

### Le calcul pas à pas : n = 1 et n = 5

Pour bien comprendre ce que fait la boucle, déroulons **une seule itération** de la simulation.

**Cas n = 1 : un échantillon d'une seule valeur**

L'instruction `np.random.exponential(5, 1)` tire **une seule** valeur, par exemple `[3.7]`. Sa moyenne est `np.mean([3.7]) = 3.7`.

La « moyenne d'un échantillon de taille 1 » est **la valeur elle-même** : il n'y a rien à lisser. En répétant 10 000 fois, on collectionne 10 000 valeurs brutes de l'exponentielle — l'histogramme est donc **exactement la distribution d'origine**, avec toute son asymétrie.

| Itération | Échantillon tiré | Moyenne |
|---|---|---|
| 1 | [3.7] | 3.7 |
| 2 | [0.4] | 0.4 |
| 3 | [11.2] | 11.2 |
| 4 | [1.9] | 1.9 |

Les moyennes varient énormément (de 0.4 à 11.2) : aucune concentration autour de 5.

**Cas n = 5 : un échantillon de cinq valeurs**

L'instruction `np.random.exponential(5, 5)` tire **cinq** valeurs, par exemple `[3.7, 0.4, 11.2, 1.9, 6.8]`. Leur moyenne est :

`(3.7 + 0.4 + 11.2 + 1.9 + 6.8) / 5 = 24.0 / 5 = 4.8`

Ici, la valeur extrême **11.2** est **compensée** par les petites valeurs (0.4 et 1.9). Le résultat, 4.8, est **plus proche de 5** (la vraie moyenne) que ne l'étaient la plupart des valeurs individuelles.

| Itération | Échantillon tiré | Moyenne |
|---|---|---|
| 1 | [3.7, 0.4, 11.2, 1.9, 6.8] | 4.8 |
| 2 | [8.1, 2.3, 1.0, 5.5, 3.6] | 4.1 |
| 3 | [0.9, 7.2, 4.4, 2.8, 9.1] | 4.9 |
| 4 | [6.0, 1.5, 3.3, 8.7, 2.0] | 4.3 |

Les moyennes (4.8, 4.1, 4.9, 4.3) sont déjà beaucoup **plus resserrées** autour de 5 que les valeurs brutes du cas n = 1. Plus `n` augmente, plus chaque moyenne agrège de valeurs, donc plus les hauts et les bas se neutralisent.

### Lecture des quatre graphiques

| Taille `n` | Forme observée | Interprétation |
|---|---|---|
| **n = 1** | Asymétrique, identique à l'exponentielle | La moyenne est la valeur elle-même : on retrouve la distribution d'origine. |
| **n = 5** | Encore asymétrique, mais déjà moins | Le moyennage commence à lisser les valeurs extrêmes, la queue se raccourcit. |
| **n = 30** | Cloche quasi symétrique | Le seuil pratique `n >= 30` est atteint : la distribution des moyennes est approximativement normale. |
| **n = 100** | Cloche normale, étroite et nette | L'approximation normale est excellente et les moyennes sont très resserrées autour de 5. |

### Pourquoi superposer une courbe rouge ?

Pour `n >= 30`, le code superpose la **courbe normale théorique** prédite par le TCL :

- **Moyenne théorique** : `mu_th = 5` — la moyenne de la population d'origine, inchangée.
- **Écart-type théorique** : `sigma_th = 5 / np.sqrt(n)` — l'**erreur standard**, soit sigma divisé par racine de n.

La courbe rouge `norm.pdf(x, mu_th, sigma_th)` épouse presque parfaitement l'histogramme : cela **confirme visuellement** que la formule du TCL prédit correctement la forme, le centre et la dispersion.

### Deux observations clés

- **La forme converge vers la normale** quelle que soit la distribution d'origine — ici une exponentielle très asymétrique. C'est le cœur du TCL.

- **La dispersion diminue quand `n` augmente** : l'axe horizontal se **rétrécit** de n = 30 (moyennes entre 2 et 8) à n = 100 (moyennes entre 4 et 7). C'est l'effet de la racine de n au dénominateur de l'erreur standard : un plus grand échantillon donne une estimation plus **précise** de la vraie moyenne.

### Le piège à éviter

Cette démonstration porte sur la distribution **des moyennes**, et non sur celle des **données individuelles**. Augmenter `n` ne rend jamais l'exponentielle elle-même normale — seules ses moyennes le deviennent. Si on traçait les 10 000 valeurs brutes au lieu de leurs moyennes, on retrouverait toujours l'asymétrie de l'exponentielle, peu importe `n`.

In [ ]:
# Démonstration visuelle du TCL
# Distribution d'origine : exponentielle (très asymétrique)
np.random.seed(42)

tailles_echantillon = [1, 5, 30, 100]
n_simulations = 10000

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, n in zip(axes, tailles_echantillon):
    # Prendre n_simulations moyennes d'échantillons de taille n
    moyennes = [np.mean(np.random.exponential(5, n)) for _ in range(n_simulations)]
    
    ax.hist(moyennes, bins=50, density=True, color='steelblue', edgecolor='white', alpha=0.7)
    ax.set_title(f'n = {n}')
    ax.set_xlabel('Moyenne')
    
    if n >= 30:
        # Superposer la courbe normale théorique
        mu_th = 5  # moyenne de l'exponentielle
        sigma_th = 5 / np.sqrt(n)
        x = np.linspace(min(moyennes), max(moyennes), 100)
        ax.plot(x, norm.pdf(x, mu_th, sigma_th), 'r-', linewidth=2)

plt.suptitle('Théorème Central Limite — Distribution exponentielle', fontsize=14)
plt.tight_layout()
plt.show()

print("Observation : même si la distribution d'origine est très asymétrique,")
print("la distribution des moyennes devient normale quand n augmente!")

## 5.2 Erreur standard et intervalle de confiance

L'**erreur standard** (SE) mesure la précision de l'estimation de la moyenne :

$$SE = \frac{\sigma}{\sqrt{n}}$$

L'**intervalle de confiance à 95%** pour la moyenne est :

$$\bar{x} \pm 1.96 \times SE$$

In [ ]:
# Calculer un intervalle de confiance
df = pd.read_csv("data/temps_attente.csv")
temps = df["temps_attente_min"]

n = len(temps)
moyenne = temps.mean()
ecart_type = temps.std()
erreur_standard = ecart_type / np.sqrt(n)

# IC à 95%
z_critique = 1.96
ic_bas = moyenne - z_critique * erreur_standard
ic_haut = moyenne + z_critique * erreur_standard

print(f"Taille de l'échantillon : {n}")
print(f"Moyenne : {moyenne:.2f} min")
print(f"Écart-type : {ecart_type:.2f} min")
print(f"Erreur standard : {erreur_standard:.2f} min")
print(f"\nIntervalle de confiance à 95% : [{ic_bas:.2f}, {ic_haut:.2f}] min")
print(f"Interprétation : on est 95% confiant que le vrai temps d'attente")
print(f"moyen est entre {ic_bas:.2f} et {ic_haut:.2f} minutes.")

In [ ]:
# Méthode rapide avec scipy
ic = stats.t.interval(confidence=0.95, df=n-1, loc=moyenne, scale=erreur_standard)
print(f"IC 95% (scipy) : [{ic[0]:.2f}, {ic[1]:.2f}] min")

# IC à 99%
ic99 = stats.t.interval(confidence=0.99, df=n-1, loc=moyenne, scale=erreur_standard)
print(f"IC 99% (scipy) : [{ic99[0]:.2f}, {ic99[1]:.2f}] min")

### Exercice 4 — TCL et intervalles de confiance

1. Générer 10 000 échantillons de taille 50 à partir d'une distribution uniforme U(0, 100)
2. Calculer la moyenne de chaque échantillon
3. Tracer l'histogramme des moyennes et vérifier qu'il est normal
4. Calculer la moyenne et l'écart-type des moyennes, et comparer avec les valeurs théoriques ($\mu = 50$, $\sigma/\sqrt{n}$)
5. En utilisant le dataset `temps_attente.csv`, calculer l'IC à 95% pour le temps d'attente moyen au **guichet** seulement (filtrer `type_service == 'Guichet'`)
6. Comparer avec l'IC du service **en ligne**. Conclusion?

In [ ]:
# Votre code ici

# 1-4. TCL avec distribution uniforme


# 5-6. IC par type de service


<details>
<summary> Cliquer pour voir la solution</summary>

```python
# 1-2. Simulation
np.random.seed(42)
moyennes = [np.mean(np.random.uniform(0, 100, 50)) for _ in range(10000)]

# 3. Histogramme
plt.figure(figsize=(10, 5))
plt.hist(moyennes, bins=50, density=True, color='steelblue', edgecolor='white', alpha=0.7)
mu_th = 50
sigma_th = (100 / np.sqrt(12)) / np.sqrt(50)  # σ_uniforme = (b-a)/√12
x = np.linspace(35, 65, 100)
plt.plot(x, norm.pdf(x, mu_th, sigma_th), 'r-', linewidth=2, label='Normale théorique')
plt.xlabel('Moyenne')
plt.ylabel('Densité')
plt.title('TCL : Moyennes d\'échantillons U(0,100), n=50')
plt.legend()
plt.show()

# 4. Comparaison
print(f"Moyenne des moyennes : {np.mean(moyennes):.2f} (théorique : 50)")
print(f"Écart-type des moyennes : {np.std(moyennes):.2f} (théorique : {sigma_th:.2f})")

# 5-6. IC par type de service
df = pd.read_csv('data/temps_attente.csv')

for service in ['Guichet', 'En ligne']:
    temps = df[df['type_service'] == service]['temps_attente_min']
    n = len(temps)
    moy = temps.mean()
    se = temps.std() / np.sqrt(n)
    ic = stats.t.interval(0.95, df=n-1, loc=moy, scale=se)
    print(f"\n{service} (n={n}) :")
    print(f"  Moyenne : {moy:.2f} min")
    print(f"  IC 95% : [{ic[0]:.2f}, {ic[1]:.2f}] min")
```

</details>

---
# Partie 6 — Comparaison des distributions

In [ ]:
# Résumé visuel des distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Bernoulli
ax = axes[0, 0]
x = [0, 1]
ax.bar(x, [0.3, 0.7], color='steelblue', edgecolor='white')
ax.set_title('Bernoulli(p=0.7)')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Échec', 'Succès'])

# Binomiale
ax = axes[0, 1]
x = np.arange(0, 21)
ax.bar(x, binom.pmf(x, 20, 0.5), color='steelblue', edgecolor='white')
ax.set_title('Binomiale(n=20, p=0.5)')

# Poisson
ax = axes[0, 2]
x = np.arange(0, 20)
ax.bar(x, poisson.pmf(x, 5), color='steelblue', edgecolor='white')
ax.set_title('Poisson(λ=5)')

# Uniforme
ax = axes[1, 0]
x = np.linspace(-1, 11, 1000)
ax.plot(x, uniform.pdf(x, 0, 10), color='steelblue', linewidth=2)
ax.fill_between(x, uniform.pdf(x, 0, 10), alpha=0.3, color='steelblue')
ax.set_title('Uniforme(0, 10)')

# Normale
ax = axes[1, 1]
x = np.linspace(-4, 4, 1000)
ax.plot(x, norm.pdf(x), color='steelblue', linewidth=2)
ax.fill_between(x, norm.pdf(x), alpha=0.3, color='steelblue')
ax.set_title('Normale(μ=0, σ=1)')

# Exponentielle
ax = axes[1, 2]
x = np.linspace(0, 20, 1000)
ax.plot(x, expon.pdf(x, scale=5), color='steelblue', linewidth=2)
ax.fill_between(x, expon.pdf(x, scale=5), alpha=0.3, color='steelblue')
ax.set_title('Exponentielle(λ=1/5)')

plt.suptitle('Comparaison des distributions de probabilité', fontsize=16)
plt.tight_layout()
plt.show()

## Récapitulatif des distributions

| Distribution | Type | Paramètres | Utilisation | scipy.stats |
|---|---|---|---|---|
| Bernoulli | Discrète | p | Succès/échec unique | `bernoulli` |
| Binomiale | Discrète | n, p | Nb succès sur n essais | `binom` |
| Poisson | Discrète | λ | Nb événements par intervalle | `poisson` |
| Uniforme | Continue | a, b | Valeurs équiprobables | `uniform` |
| Normale | Continue | μ, σ | Données naturelles, TCL | `norm` |
| Exponentielle | Continue | λ | Temps d'attente | `expon` |

---

## Prochaine séance

**Séance 6 — Tests d'hypothèses**
- Formulation d'hypothèses (H₀ et H₁)
- p-value et seuil de significativité
- t-test, chi², ANOVA
- Intervalles de confiance

**Date :** Jeudi 28 mai 2026, 18h30

---

*FSCI 6222 — Université de Moncton — Été 2026*